# Demo: Metoddagen 11/9

In [4]:
# Inställning av LLM

import openai

def LLM_API(api_key, variables, input_config = None, model="", message_content=""):

    client = openai.OpenAI(
        api_key = api_key,
        base_url = "https://llm.lab.sspcloud.fr/api"
    )

    prompt_content = f"{message_content}, does the above-mentioned text, relating to a web-page, contain a {variables}?"

    response = client.chat.completions.create(
        model = model,
        messages = [
            {"role":"system", "max_completion_tokens":"1", "content":"You are an expert about classifying texts, your answer is a one-word response, chosen from [Yes, No]."},
            {"role":"user", "content": prompt_content},
        ],
        temperature = 0.0,
        max_completion_tokens = 1,
        max_tokens = 1
    )

    return response.choices[0].message.content

In [2]:
# Test
import requests 
import justext
response = requests.get("https://www.scania.se")

print(response.content)

language = "Swedish"
extracted_paragraphs = justext.justext(response.content, justext.get_stoplist(language))
print(extracted_paragraphs[0].__dict__)
print(extracted_paragraphs[0].class_type)

print([paragraph.text for paragraph in extracted_paragraphs if paragraph.class_type in ["good", "neargood"]])

b'<!DOCTYPE html>\n\n  <html dir="ltr" lang="sv-SE">\n    <head>\n  <meta http-equiv="X-UA-Compatible" content="IE=edge"/>\n  <meta charset="utf-8"/>\n  <meta name="viewport" content="width=device-width, initial-scale=1"/>\n  <meta http-equiv="content-encoding" content="text/html"/>\n  \n  <script defer="defer" type="text/javascript" src="/.rum/@adobe/helix-rum-js@%5E2/dist/rum-standalone.js"></script>\n<script src="https://cdn.cookielaw.org/scripttemplates/otSDKStub.js" data-document-language="true" type="text/javascript" charset="UTF-8" data-domain-script="9d78e14a-5af5-4fc7-8d13-be82ad0bce2f"></script>\n  <script type="text/javascript">\n    function OptanonWrapper() {    \n      const tagsH2 = document.querySelectorAll(\'#onetrust-consent-sdk h2\');\n      for (let i = 0; i < tagsH2.length; i++) { otReplace(tagsH2[i], "ot-h2"); }\n      const tagsH3 = document.querySelectorAll(\'#onetrust-consent-sdk h3\');\n      for (let i = 0; i < tagsH3.length; i++) { otReplace(tagsH3[i], "ot-h

In [5]:
import requests
import justext
import os
import s3fs
import json
import random
import pandas as pd

path_api_key = "/home/onyxia/work/SCB/Metoddagen/api_key.txt"
path_variables = "/home/onyxia/work/SCB/Metoddagen/variables.txt"
path_url_data = "/home/onyxia/work/SCB/Metoddagen/url_data.csv"
output_path = "/home/onyxia/work/SCB/Metoddagen/output_all_variables.json"

# Read API Key
with open(path_api_key, "r", encoding="utf-8") as file:
    api_key = file.readlines()[0]

# Read the variables
with open(path_variables, "r", encoding="utf-8") as file:
    variables = [v.strip() for v in file.readlines()]

# Read the url_data
url_data = pd.read_csv(path_url_data)[['url', 'country']].values.tolist()

# Samlad struktur för alla variabler
all_results = {}

for var in variables:
    job_vacancies = {}
    print("Variable:", var)

    for url, country in url_data:
        original_url = url  # for logging
        if not url.startswith("http"):
            url = "https://" + url

        try:
            response = requests.get(url, timeout=10)
        except Exception as e:
            job_vacancies[original_url] = f"Request failed: {str(e)}"
            continue

        if not (200 <= response.status_code < 400):
            job_vacancies[original_url] = f"Bad status code: {response.status_code}"
            continue

        language = {
            "DE": "German",
            "NL": "Dutch",
            "PL": "Polish",
            "AT": "German",
            "SE": "Swedish"
        }.get(country, "English")

        extracted_paragraphs = justext.justext(response.content, justext.get_stoplist(language))
        extracted_text = "".join([
            paragraph.text for paragraph in extracted_paragraphs
            if (paragraph.cf_class in ["good", "neargood"] or paragraph.class_type in ["good", "neargood"])
        ])

        if len(extracted_text.strip()) < 1:
            job_vacancies[original_url] = f"No extractable text found (language: {language})"
            continue

        response_text = LLM_API(api_key, var, None, "magistral:latest", extracted_text).strip()

        if response_text in ["Yes", "No"]:
            job_vacancies[original_url] = response_text
        else:
            job_vacancies[original_url] = f"LLM response unclear or unexpected: '{response_text}'"

    all_results[var] = job_vacancies

# Spara allt i en enda JSON-fil
with open(output_path, "w", encoding="utf-8") as fp:
    json.dump(all_results, fp, ensure_ascii=False, indent=2)

print("Done!")


Variable: hotellverksamhet


APIConnectionError: Connection error.